In [ ]:
# Importación de librerías esenciales para el preprocesamiento analítico
import pandas as pd # Manipulación y estructuración analítica de datos tabulares
import numpy as np # Operaciones matemáticas y transformaciones vectoriales nativas
from sklearn.model_selection import train_test_split # Partición dinámica de conjuntos de evaluación
from sklearn.preprocessing import StandardScaler # Normalización estadística de variables continuas

# Cargar el archivo original de datos de forma dinámica desde almacenamiento local
df = pd.read_csv('dataset/Teen_Mental_Health_Dataset.csv')

# Inspección de las dimensiones iniciales de la estructura cargada
df.shape


(1200, 13)

In [ ]:
# 1. Separar características de entrada (X) eliminando el target y la columna ruidosa
X = df.drop(columns=['academic_performance', 'depression_label'])

# Aislamiento de la variable predictiva objetivo (target)
y = df['academic_performance']

# 2. Análisis analítico dinámico del desbalance de clases usando funciones nativas de Pandas
distribucion_absoluta = df['depression_label'].value_counts()

# Cuantificación relativa porcentual para la evaluación de sesgos en la distribución
distribucion_porcentual = df['depression_label'].value_counts(normalize=True) * 100

# Agrupar dinámicamente en un DataFrame descriptivo para su renderización en el Notebook
pd.DataFrame({
    'Frecuencia Absoluta': distribucion_absoluta,
    'Proporción Porcentual (%)': distribucion_porcentual.round(2)
})


,Frecuencia Absoluta,Proporción Porcentual (%)
depression_label,,
0,1169,97.42
1,31,2.58


In [ ]:
# Dividir el conjunto de características garantizando reproducibilidad matemática mediante random_state
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Forzar copias explícitas en memoria para evitar alertas de asignación de fragmentos (SettingWithCopyWarning)
X_train = X_train.copy()
X_test = X_test.copy()

# Mostrar las dimensiones de las matrices generadas post-partición
print("Dimensiones de los sets de entrenamiento y prueba:")
print(X_train.shape, X_test.shape)

# Validación dinámica de la integridad estructural de los tipos de datos en entrenamiento
X_train.dtypes


,0
age,int64
gender,object
daily_social_media_hours,float64
platform_usage,object
sleep_hours,float64
screen_time_before_sleep,float64
physical_activity,float64
social_interaction_level,object
stress_level,int64
anxiety_level,int64


In [ ]:
# 1. Ejecutar el mapeo ordinal jerárquico definiendo la escala de transformación
nivel_map = {'low': 0, 'medium': 1, 'high': 2}

# Aplicación de la regla de mapeo sobre la matriz de entrenamiento y prueba
X_train['social_interaction_level'] = X_train['social_interaction_level'].map(nivel_map)
X_test['social_interaction_level'] = X_test['social_interaction_level'].map(nivel_map)

# 2. Aplicar codificación One-Hot y alinear de forma dinámica las columnas de ambos sets
X_train_encoded = pd.get_dummies(X_train, columns=['gender', 'platform_usage'], drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=['gender', 'platform_usage'], drop_first=True)

# Sincronización topológica de columnas para prevenir discrepancias dimensionales entre sets
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# 3. Forzar la conversión de tipos booleanos nativos de pandas a enteros (1 / 0) para compatibilidad algorítmica
for col in X_train_encoded.columns:
    if X_train_encoded[col].dtype == bool or X_train_encoded[col].dtype == 'bool':
        X_train_encoded[col] = X_train_encoded[col].astype(int)
        X_test_encoded[col] = X_test_encoded[col].astype(int)

# 4. VALIDACIÓN VISUAL DINÁMICA: Comprobación de que todos los objetos mutaron a dtypes numéricos
X_train_encoded.dtypes


,0
age,int64
daily_social_media_hours,float64
sleep_hours,float64
screen_time_before_sleep,float64
physical_activity,float64
social_interaction_level,int64
stress_level,int64
anxiety_level,int64
addiction_level,int64
gender_male,int64


Aquí se separan las columnas numéricas que serán escaladas de las columnas binarias. El escalado se realiza solo con las estadísticas del conjunto de entrenamiento para evitar fuga de datos hacia el conjunto de prueba.

In [ ]:
# 1. Agrupar las columnas numéricas continuas/escalares a estandarizar
columnas_numericas = ['age', 'daily_social_media_hours', 'sleep_hours',
                     'screen_time_before_sleep', 'physical_activity',
                     'social_interaction_level', 'stress_level',
                     'anxiety_level', 'addiction_level']

# Identificar dinámicamente las columnas binarias que deben permanecer intactas post-codificación
columnas_binarias = [col for col in X_train_encoded.columns if col not in columnas_numericas]

# 2. Instanciar y ajustar el escalador únicamente con estadísticas de entrenamiento (prevención de data leakage)
scaler = StandardScaler()

# Generación de copias independientes para almacenar los resultados transformados
X_train_ready = X_train_encoded.copy()
X_test_ready = X_test_encoded.copy()

# Transformación de las variables escalares proyectando la misma escala hacia el test
X_train_ready[columnas_numericas] = scaler.fit_transform(X_train_encoded[columnas_numericas])
X_test_ready[columnas_numericas] = scaler.transform(X_test_encoded[columnas_numericas])

# 3. VALIDACIÓN ESTADÍSTICA DINÁMICA: Comprobar agregaciones de variables numéricas escaladas
# Nota: La media se expresará en notación científica extremadamente cercana a cero, y la desviación estándar será exactamente 1.00
X_train_ready[columnas_numericas].agg(['mean', 'std']).round(4)


,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level
mean,-0.0000,0.0000,0.0000,0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000
std,1.0005,1.0005,1.0005,1.0005,1.0005,1.0005,1.0005,1.0005,1.0005


In [ ]:
# Comprobación analítica de que las variables del One-Hot no sufrieron alteraciones de rango decimal
# Se espera que el valor mínimo sea 0 y el máximo sea 1
X_train_ready[columnas_binarias].agg(['min', 'max'])


,gender_male,platform_usage_Instagram,platform_usage_TikTok
min,0,0,0
max,1,1,1


Antes de exportar los datos finales, añadimos la columna objetivo en cada conjunto limpio. Esto permite guardar los archivos de entrenamiento y prueba en formato CSV listos para su modelado posterior.

In [ ]:
# 1. Integrar la variable predictiva y_train e y_test a las estructuras preprocesadas finales
train_final_export = X_train_ready.copy()
train_final_export['target_academic_performance'] = y_train

test_final_export = X_test_ready.copy()
test_final_export['target_academic_performance'] = y_test

# 2. Exportar los sets de datos limpios a archivos CSV locales omitiendo el índice secuencial
train_final_export.to_csv('dataset/teen_train_clean.csv', index=False)
test_final_export.to_csv('dataset/teen_test_clean.csv', index=False)

# 3. Renderizar una vista previa dinámica de los primeros registros limpios en el Notebook
train_final_export.head(5)


,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,gender_male,platform_usage_Instagram,platform_usage_TikTok,target_academic_performance
331,0.020422,0.766421,1.135845,-0.476546,1.029629,-1.195656,1.564665,-1.598783,-0.553219,0,1,0,3.27
409,-0.469701,0.569455,0.720924,0.915253,0.513021,0.043901,1.564665,1.560029,-1.256185,0,1,0,2.84
76,-1.449946,-1.695645,0.444311,0.080173,-0.347993,1.283458,-0.141854,-0.194867,1.555678,1,0,1,2.55
868,0.020422,0.372490,-1.491985,-0.337366,-1.209007,1.283458,0.540753,0.507092,1.555678,0,0,1,2.54
138,-0.959824,0.914145,-0.316377,-0.894086,-0.175790,0.043901,0.882057,-1.247804,-0.904702,0,1,0,3.81
